# Proyecto Final: Modelado Inicial de Indicadores Socioeconómicos en Latinoamérica

**Curso:** CC3074 - Minería de Datos  
**Framework:** CRISP-DM - Fase 4: Modelado (porción inicial)  
**Fuente de datos:** CEPALSTAT - Observatorio de Desarrollo Digital (ODD)  
**Indicador:** Personas usuarias de Internet por grupo etario, países seleccionados de América Latina y el Caribe  

**Integrantes:**

- Cristian Túnchez (231359)    
- Dulce Ambrosio (231143)  
- Daniel Chet (231177)  
- Javier Linares (231135)  


## Semana 3 - Modelado Inicial: Construcción y Comparación de Modelos Base

---
### Contexto y Objetivo

Partiendo del conjunto `data_transformado.csv` (175 filas × 60 columnas) construido previamente, esta semana corresponde a la **porción inicial de la Fase 4 (Modeling) del marco CRISP-DM**. El trabajo se centra en:

1. **Formular** el problema como una tarea supervisada con variable objetivo justificada.
2. **Diseñar** una prueba robusta evitando data leakage tanto en features como en el escalado.
3. **Entrenar** varios modelos de familias distintas (lineal, basada en distancia, basada en árboles).
4. **Comparar** mediante métricas múltiples (MAE, RMSE, R², MAPE) en train y test, diagnosticando overfit.
5. **Diagnosticar** visualmente residuales, errores por país/año, e importancia/coeficientes de features.
6. **Documentar** los hallazgos del ranking inicial.

Este notebook usa **hiperparámetros por defecto** y un **único split train/test** para producir una primera lectura comparativa de los modelos base.


**Configuración del entorno e importación de librerías**

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats as sp_stats
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    mean_absolute_percentage_error,
)
import warnings

warnings.filterwarnings('ignore')

# Configuración visual coherente con el resto del proyecto
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size'] = 11
plt.rcParams['axes.titlesize'] = 14
plt.rcParams['axes.labelsize'] = 12
sns.set_style('whitegrid')
sns.set_palette('Set2')

# Semilla global de reproducibilidad
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print('Bibliotecas cargadas correctamente.')
print(f'Semilla global de reproducibilidad: RANDOM_STATE = {RANDOM_STATE}')

Bibliotecas cargadas correctamente.
Semilla global de reproducibilidad: RANDOM_STATE = 42


---
### 1. Formulación del Problema de Modelado

#### 1.1 Tipo de problema: regresión supervisada

El proyecto plantea como pregunta de negocio: **¿es posible predecir el uso de Internet en un país a partir de su historia reciente y de los patrones intergeneracionales?**. Esto se traduce en un problema de **regresión supervisada**, dado que:

- El target es **continuo** (porcentaje en el intervalo físico [0, 100]).
- Se dispone de una muestra etiquetada con valores conocidos de `pct_uso_total`.
- El objetivo es estimar el valor numérico, no una clase ni un agrupamiento.

#### 1.2 Justificación de la variable objetivo

Se selecciona **`pct_uso_total`** como variable objetivo. Razones:

1. **Relevancia del dominio**: es el indicador principal de adopción digital agregada, el más usado por organismos como CEPAL y la UIT para comparar países.
2. **Cobertura**: presente en todas las 175 filas del dataset (sin NaN tras la preparación previa).
3. **Rango y variabilidad**: media 46.59 %, desviación estándar 21.39, mínimo 5.0, máximo 88.5; suficiente variabilidad para que la regresión tenga sentido (no se trata de una constante).
4. **Distribución**: aproximadamente simétrica según el EDA (sin outliers IQR), lo que es compatible con los supuestos de modelos lineales sin transformación previa.
5. **Interpretación práctica**: predecir el nivel agregado de adopción permite a tomadores de decisión proyectar inversiones en infraestructura digital, comparar trayectorias entre países y detectar rezagos.

#### 1.3 Enfoque predictivo (lag-based)

Una decisión clave en regresión sobre datos de panel es **qué información se permite al modelo en el momento de la predicción**. El dataset ofrece dos enfoques posibles:

| Enfoque | Predictoras admitidas | Riesgo |
|---|---|---|
| **Contemporáneo** | Indicadores del mismo año (`pct_uso_<=17`, `pct_uso_18_25`, ...) | Composición trivial: el total es función directa de los segmentos por edad. R² artificialmente alto, sin valor predictivo real. |
| **Predictivo (lag-based)** | Solo información del **año anterior** (`*_lag1`) + atributos invariantes (país, tiempo) | Forecasting realista: el modelo aprende a proyectar el próximo año a partir del histórico. |

Se elige el **enfoque predictivo (lag-based)** por su valor práctico y por evitar leakage estructural. Concretamente, se excluyen:

- `pct_uso_total` mismo (target).
- Las cinco columnas raw del mismo año (`pct_uso_<=17`, `pct_uso_18_25`, `pct_uso_26_50`, `pct_uso_51_65`, `pct_uso_66_mas`).
- Cualquier columna derivada del target del mismo año: `brecha_total_mayor`, `brecha_joven_total`, `delta_pct_uso_total`, `categoria_adopcion` y sus versiones estandarizadas.

---
### 2. Carga del Dataset Transformado

**Lectura del archivo `data_transformado.csv`**

In [ ]:
df = pd.read_csv('data_transformado.csv')

print(f'Dataset cargado: {df.shape[0]} filas x {df.shape[1]} columnas')
print(f'Rango de años:   {df["año"].min()} - {df["año"].max()}')
print(f'Países:          {df["pais"].nunique()}')

print(f'\nPrimeras filas (columnas principales):')
display(df[['pais', 'año', 'pct_uso_total', 'categoria_adopcion']].head())

Dataset cargado: 175 filas x 60 columnas
Rango de años:   2000 - 2022
Países:          14

Primeras filas (columnas principales):


,pais,año,pct_uso_total,categoria_adopcion
0,Argentina,2016,71.1,Avanzada
1,Argentina,2017,74.4,Avanzada
2,Argentina,2018,77.7,Madura
3,Argentina,2019,80.0,Madura
4,Argentina,2020,85.6,Madura


**Inspección de tipos por bloque y conteo de nulos**

In [ ]:
bloques = {
    'Llaves identificadoras (pais, año)':            sum(1 for c in df.columns if c in ('pais', 'año')),
    'Indicadores raw (pct_uso_*)':                   sum(1 for c in df.columns if c.startswith('pct_uso') and '_lag' not in c and not c.endswith('_z')),
    'Brechas (brecha_*)':                            sum(1 for c in df.columns if c.startswith('brecha_') and not c.endswith('_z')),
    'Lags temporales (*_lag1)':                      sum(1 for c in df.columns if c.endswith('_lag1')),
    'Deltas anuales (delta_*)':                      sum(1 for c in df.columns if c.startswith('delta_') and not c.endswith('_z')),
    'Variables temporales (años_*)':                 sum(1 for c in df.columns if c.startswith('años_')),
    'Categórica discretizada (categoria_adopcion)':  sum(1 for c in df.columns if c == 'categoria_adopcion'),
    'Dummies de país (pais_*)':                      sum(1 for c in df.columns if c.startswith('pais_')),
    'Estandarizadas (*_z)':                          sum(1 for c in df.columns if c.endswith('_z')),
}

print('Composición de columnas del dataset transformado:')
for bloque, n in bloques.items():
    print(f'  - {bloque:50s} {n:3d}')
print(f'  {"─" * 55}')
print(f'  - {"TOTAL":50s} {sum(bloques.values()):3d}')

print(f'\nNaN por columna (solo columnas afectadas):')
nans = df.isna().sum()
display(nans[nans > 0].to_frame('n_nan'))

Composición de columnas del dataset transformado:
  - Llaves identificadoras (pais, año)                   2
  - Indicadores raw (pct_uso_*)                          6
  - Brechas (brecha_*)                                   3
  - Lags temporales (*_lag1)                             6
  - Deltas anuales (delta_*)                             6
  - Variables temporales (años_*)                        2
  - Categórica discretizada (categoria_adopcion)         1
  - Dummies de país (pais_*)                            13
  - Estandarizadas (*_z)                                21
  ───────────────────────────────────────────────────────
  - TOTAL                                               60

NaN por columna (solo columnas afectadas):


,n_nan
pct_uso_<=17_lag1,14
pct_uso_18_25_lag1,14
pct_uso_26_50_lag1,14
pct_uso_51_65_lag1,14
pct_uso_66_mas_lag1,14
pct_uso_total_lag1,14
delta_pct_uso_<=17,14
delta_pct_uso_18_25,14
delta_pct_uso_26_50,14
delta_pct_uso_51_65,14


Las 24 columnas con NaN corresponden todas a derivaciones del primer año observado por país (lags y deltas, en sus versiones raw y `_z`). Esto define cuáles son las filas que se eliminarán al construir el dataset modelable.

---
### 3. Selección de Features

#### 3.1 Identificación de leakage (features prohibidas)

Antes de seleccionar predictoras, se enumeran explícitamente las columnas que **no pueden** usarse porque son funciones determinísticas del target o información del mismo año que el target. Incluirlas produciría un R² engañosamente alto y un modelo sin valor práctico.

In [ ]:
LEAKAGE_DIRECTO = {
    'pct_uso_total':           'Target — no puede ser feature de sí mismo.',
    'pct_uso_total_z':         'Estandarización del target.',
    'brecha_total_mayor':      'Definida como pct_uso_total - pct_uso_66_mas.',
    'brecha_joven_total':      'Definida como pct_uso_18_25 - pct_uso_total.',
    'delta_pct_uso_total':     'Definida como pct_uso_total - pct_uso_total_lag1.',
    'categoria_adopcion':      'Discretización (pd.cut) del target.',
    'brecha_total_mayor_z':    'Estandarización de brecha derivada del target.',
    'brecha_joven_total_z':    'Estandarización de brecha derivada del target.',
    'delta_pct_uso_total_z':   'Estandarización de delta derivada del target.',
}

LEAKAGE_CONTEMPORANEO = {
    'pct_uso_<=17':    'Indicador del mismo año — composición trivial del total.',
    'pct_uso_18_25':   'Indicador del mismo año — composición trivial del total.',
    'pct_uso_26_50':   'Indicador del mismo año — composición trivial del total.',
    'pct_uso_51_65':   'Indicador del mismo año — composición trivial del total.',
    'pct_uso_66_mas':  'Indicador del mismo año — composición trivial del total.',
    'pct_uso_<=17_z':  'Estandarización de indicador contemporáneo.',
    'pct_uso_18_25_z': 'Estandarización de indicador contemporáneo.',
    'pct_uso_26_50_z': 'Estandarización de indicador contemporáneo.',
    'pct_uso_51_65_z': 'Estandarización de indicador contemporáneo.',
    'pct_uso_66_mas_z':'Estandarización de indicador contemporáneo.',
    'delta_pct_uso_<=17':  'Delta contemporáneo del segmento.',
    'delta_pct_uso_18_25': 'Delta contemporáneo del segmento.',
    'delta_pct_uso_26_50': 'Delta contemporáneo del segmento.',
    'delta_pct_uso_51_65': 'Delta contemporáneo del segmento.',
    'delta_pct_uso_66_mas':'Delta contemporáneo del segmento.',
    'delta_pct_uso_<=17_z':  'Estandarización de delta contemporáneo.',
    'delta_pct_uso_18_25_z': 'Estandarización de delta contemporáneo.',
    'delta_pct_uso_26_50_z': 'Estandarización de delta contemporáneo.',
    'delta_pct_uso_51_65_z': 'Estandarización de delta contemporáneo.',
    'delta_pct_uso_66_mas_z':'Estandarización de delta contemporáneo.',
}

tabla_leakage = pd.DataFrame({
    'columna':   list(LEAKAGE_DIRECTO.keys()) + list(LEAKAGE_CONTEMPORANEO.keys()),
    'tipo':      ['directo'] * len(LEAKAGE_DIRECTO) + ['contemporáneo'] * len(LEAKAGE_CONTEMPORANEO),
    'motivo':    list(LEAKAGE_DIRECTO.values()) + list(LEAKAGE_CONTEMPORANEO.values()),
})

print(f'Columnas excluidas por leakage: {len(tabla_leakage)}')
print(f'  - Leakage directo (función del target):  {len(LEAKAGE_DIRECTO)}')
print(f'  - Leakage contemporáneo (mismo año):     {len(LEAKAGE_CONTEMPORANEO)}')

display(tabla_leakage)

Columnas excluidas por leakage: 29
  - Leakage directo (función del target):  9
  - Leakage contemporáneo (mismo año):     20


,columna,tipo,motivo
0,pct_uso_total,directo,Target — no puede ser feature de sí mismo.
1,pct_uso_total_z,directo,Estandarización del target.
2,brecha_total_mayor,directo,Definida como pct_uso_total - pct_uso_66_mas.
3,brecha_joven_total,directo,Definida como pct_uso_18_25 - pct_uso_total.
4,delta_pct_uso_total,directo,Definida como pct_uso_total - pct_uso_total_lag1.
5,categoria_adopcion,directo,Discretización (pd.cut) del target.
6,brecha_total_mayor_z,directo,Estandarización de brecha derivada del target.
7,brecha_joven_total_z,directo,Estandarización de brecha derivada del target.
8,delta_pct_uso_total_z,directo,Estandarización de delta derivada del target.
9,pct_uso_<=17,contemporáneo,Indicador del mismo año — composición trivial ...


De las 60 columnas del dataset, 29 quedan descartadas como predictoras: 9 por leakage directo (incluyendo el propio target) y 20 por compartir año con el target. El conjunto admisible se reduce así a 31 columnas, sobre las cuales se aplica el criterio lag-based para llegar al subconjunto definitivo.

#### 3.2 Conjunto final de predictoras (enfoque lag-based)

Las predictoras finales corresponden estrictamente a información disponible **antes** del año a predecir, más atributos invariantes:

In [ ]:
LAGS_RAW = [
    'pct_uso_<=17_lag1',
    'pct_uso_18_25_lag1',
    'pct_uso_26_50_lag1',
    'pct_uso_51_65_lag1',
    'pct_uso_66_mas_lag1',
    'pct_uso_total_lag1',
]
BRECHA_INDEPENDIENTE = ['brecha_joven_mayor']  # pct_uso_18_25 - pct_uso_66_mas; no involucra el target
TEMPORALES = ['años_desde_2000', 'años_desde_primer_registro_pais']
DUMMIES_PAIS = sorted([c for c in df.columns if c.startswith('pais_')])

FEATURES = LAGS_RAW + BRECHA_INDEPENDIENTE + TEMPORALES + DUMMIES_PAIS
TARGET = 'pct_uso_total'

resumen_features = pd.DataFrame({
    'grupo': [
        'Lags temporales (otros segmentos + total, t-1)',
        'Brecha intergeneracional independiente del target',
        'Variables temporales',
        'Dummies de país (one-hot, 13 = 14 países - 1 referencia)',
    ],
    'cantidad': [len(LAGS_RAW), len(BRECHA_INDEPENDIENTE), len(TEMPORALES), len(DUMMIES_PAIS)],
    'features': [
        ', '.join(LAGS_RAW),
        ', '.join(BRECHA_INDEPENDIENTE),
        ', '.join(TEMPORALES),
        f'13 dummies (referencia: Argentina, eliminado por drop_first=True)',
    ],
})

print(f'Total de predictoras: {len(FEATURES)}')
print(f'Variable objetivo:    {TARGET}\n')
display(resumen_features)

Total de predictoras: 22
Variable objetivo:    pct_uso_total



,grupo,cantidad,features
0,"Lags temporales (otros segmentos + total, t-1)",6,"pct_uso_<=17_lag1, pct_uso_18_25_lag1, pct_uso..."
1,Brecha intergeneracional independiente del target,1,brecha_joven_mayor
2,Variables temporales,2,"años_desde_2000, años_desde_primer_registro_pais"
3,"Dummies de país (one-hot, 13 = 14 países - 1 r...",13,"13 dummies (referencia: Argentina, eliminado p..."


#### 3.3 Manejo de NaN y construcción del dataset de modelado

Los seis lags introducen 14 NaN por columna (un NaN por país en su primer año observado). Para entrenar modelos que no admiten NaN nativamente, se eliminan **exactamente esas 14 filas**: 175 → 161 filas.

In [ ]:
n_inicial = len(df)
mascara_completa = df[FEATURES + [TARGET]].notna().all(axis=1)
df_modelo = (
    df.loc[mascara_completa, ['pais', 'año'] + FEATURES + [TARGET]]
      .copy()
      .reset_index(drop=True)
)
n_final = len(df_modelo)

print(f'Filas antes del drop:                       {n_inicial}')
print(f'Filas eliminadas (primer año por país):     {n_inicial - n_final}')
print(f'Filas finales para modelado:                {n_final}')

assert df_modelo[FEATURES + [TARGET]].isna().sum().sum() == 0, 'No deben quedar NaN tras el drop'
print(f'\nDataset listo: sin NaN en features ni target. Forma final: {df_modelo[FEATURES + [TARGET]].shape}')

print(f'\nPrimeras filas del dataset de modelado (subset de columnas):')
muestra_cols = ['pais', 'año', 'pct_uso_total_lag1', 'brecha_joven_mayor', 'años_desde_2000', 'pct_uso_total']
display(df_modelo[muestra_cols].head())

Filas antes del drop:                       175
Filas eliminadas (primer año por país):     14
Filas finales para modelado:                161

Dataset listo: sin NaN en features ni target. Forma final: (161, 23)

Primeras filas del dataset de modelado (subset de columnas):


,pais,año,pct_uso_total_lag1,brecha_joven_mayor,años_desde_2000,pct_uso_total
0,Argentina,2017,71.1,55.7,17,74.4
1,Argentina,2018,74.4,50.5,18,77.7
2,Argentina,2019,77.7,45.9,19,80.0
3,Argentina,2020,80.0,40.5,20,85.6
4,Argentina,2021,85.6,37.5,21,87.2
